<a href="https://colab.research.google.com/github/amiralito/Helixer_colab/blob/main/Helixer_gene_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Helixer — batch gene prediction + BUSCO on Google Colab

[Helixer](https://github.com/usadellab/Helixer) is a deep-learning *ab initio* gene finder (a convolutional + bidirectional-LSTM network that predicts base-wise genic classes and coding phase, decoded into gene models by the HMM-based post-processor [HelixerPost](https://github.com/usadellab/HelixerPost)). It predicts gene structures from a **genome FASTA only** and writes a **GFF3** with genes, mRNAs, exons, CDS and UTRs. This notebook runs Helixer over **one or many genomes**, extracts **CDS + protein FASTA** with `gffread`, gzips everything, then scores completeness with **BUSCO** and compiles the results into a table.

**How to use:** set a GPU runtime (*Runtime > Change runtime type > GPU*; a free T4 is enough, an L4/A100 is faster), then fill the form fields and run each cell top to bottom. **The code is hidden** — to edit a cell, open its **menu (vertical dots) > Show code**.

Mount Drive in cell 4 to read genomes from a folder, cache the models, and keep results across sessions. Outputs are named after each genome (`<genome>_gff.gff3.gz`, `<genome>_cds.fasta.gz`, `<genome>_protein.fasta.gz`); gene IDs take the form `<SPECIES>_<seqid>_000001` (default `SPECIES = Helixer`).

In [ ]:
#@title 1 · Check runtime — GPU { display-mode: "form" }
import sys, subprocess
print("Colab Python:", sys.version.split()[0], " (Helixer runs in its own Python 3.10 env, built in cell 2)")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    smi = ""
if smi:
    print("GPU   :", smi)
    GPU_CC = float(smi.split(",")[2])
    GPU_MEM_GB = float(smi.split(",")[1].strip().split()[0]) / 1024
    if GPU_CC >= 10.0:
        print("WARNING: Blackwell-class GPU. Helixer's pinned stack (TensorFlow 2.15, cuDNN 8.9) predates it;\n"
              "         if cell 3's GPU smoke test fails, switch to a T4 / L4 / A100 / H100 runtime.")
else:
    GPU_CC, GPU_MEM_GB = 0.0, 0.0
    print("No GPU - Helixer will run on the CPU, which is only practical for a few Mb of sequence.\n"
          "Runtime > Change runtime type > GPU.")

In [ ]:
#@title 2 · Install Helixer + HelixerPost { display-mode: "form" }
#@markdown Helixer is pinned to **TensorFlow 2.15 / Keras 2** and needs `tensorflow-addons`, which has no
#@markdown wheels for Colab's Python 3.12. Rather than fight Colab's own TensorFlow, this builds a
#@markdown **self-contained Python 3.10 environment beside it** (Helixer's tested requirements + CUDA 12.2 /
#@markdown cuDNN 8.9 from pip) and runs Helixer from there. Colab's stack is untouched; no restart needed.
#@markdown The Rust post-processor **HelixerPost** is compiled from source. ~4-6 min, ~5 GB of local disk.
HELIXER_VERSION = "v0.3.7"  #@param {type:"string"}
#@markdown A Helixer release tag (or `main`). Cell 8 records it in `run_config.json`.
import os, sys, glob, shutil, subprocess

REPO = "/content/Helixer"
POST_REPO = "/content/HelixerPost"
ENVDIR = "/content/helixer-env"
ENV_BIN = os.path.join(ENVDIR, "bin")
HELIXER_PY = os.path.join(ENV_BIN, "python")
POST_BIN = os.path.join(ENV_BIN, "helixer_post_bin")
TF_SPEC = "tensorflow[and-cuda]==2.15.1"      # the TF in requirements.3.10.txt, plus its CUDA 12.2 wheels
UV = [sys.executable, "-m", "uv"]

def sh(*cmd, **kw):
    print("$", " ".join(str(c) for c in cmd))
    if subprocess.call([str(c) for c in cmd], **kw) != 0:
        raise RuntimeError(" ".join(str(c) for c in cmd))

def apt(*pkgs):
    cmd = "apt-get -qq install -y " + " ".join(pkgs) + " > /dev/null"
    if subprocess.call(cmd, shell=True) != 0:          # stale package index on some images
        subprocess.call("apt-get -qq update > /dev/null", shell=True)
        sh("bash", "-c", cmd)

# gffread (cell 8) + HDF5 headers / pkg-config (HelixerPost build)
apt("gffread", "libhdf5-dev", "pkg-config")

if not os.path.isdir(REPO):
    sh("git", "-c", "advice.detachedHead=false", "clone", "-q", "--depth", "1", "--branch", HELIXER_VERSION,
       "https://github.com/usadellab/Helixer.git", REPO)

ready = os.path.exists(HELIXER_PY) and subprocess.call(
    [HELIXER_PY, "-c", "import tensorflow, tensorflow_addons, helixer"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0

if ready:
    print("environment already built:", ENVDIR)
else:
    sh(sys.executable, "-m", "pip", "install", "-q", "uv")
    sh(*UV, "python", "install", "-q", "3.10")
    sh(*UV, "venv", "-q", ENVDIR, "--python", "3.10")
    pip = [*UV, "pip", "install", "-q", "--python", HELIXER_PY]
    sh(*pip, "-r", os.path.join(REPO, "requirements.3.10.txt"))   # Helixer's tested pins (TF 2.15.1, Keras 2.15)
    sh(*pip, REPO)                                                  # Helixer + GeenuFF
    sh(*pip, TF_SPEC)                                               # CUDA 12.2 / cuDNN 8.9 wheels for TF
    subprocess.call([*UV, "cache", "clean", "-q"])

if not os.path.exists(POST_BIN):
    cargo = shutil.which("cargo") or os.path.expanduser("~/.cargo/bin/cargo")
    if not os.path.exists(cargo):
        sh("bash", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q --profile minimal")
    if not os.path.isdir(POST_REPO):
        sh("git", "clone", "-q", "--depth", "1", "https://github.com/usadellab/HelixerPost.git", POST_REPO)
    sh(cargo, "build", "-q", "--release", "--manifest-path", os.path.join(POST_REPO, "Cargo.toml"),
       env=dict(os.environ, RUSTFLAGS="-Awarnings"))
    shutil.copy(os.path.join(POST_REPO, "target", "release", "helixer_post_bin"), POST_BIN)

# Environment used for every Helixer call (cells 3 and 8). Colab's PYTHONPATH and inline matplotlib
# backend would leak Python 3.12 packages into the 3.10 env, so both are dropped/overridden; the pip
# CUDA libraries go first on the library path, ptxas/libdevice are exposed for XLA.
SITE = subprocess.check_output([HELIXER_PY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               text=True).strip()
NV = os.path.join(SITE, "nvidia")
HELIXER_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV")}
HELIXER_ENV.update(
    PATH=":".join([ENV_BIN, os.path.join(NV, "cuda_nvcc", "bin"), os.environ.get("PATH", "")]),
    LD_LIBRARY_PATH=":".join(sorted(glob.glob(os.path.join(NV, "*", "lib"))) + [os.environ.get("LD_LIBRARY_PATH", "")]),
    XLA_FLAGS=f"--xla_gpu_cuda_data_dir={os.path.join(NV, 'cuda_nvcc')}",
    MPLBACKEND="Agg", TF_CPP_MIN_LOG_LEVEL="2", TF_FORCE_GPU_ALLOW_GROWTH="true",
    VIRTUAL_ENV=ENVDIR,
)
print("\nDone. Helixer will run with:", HELIXER_PY)

In [ ]:
#@title 3 · Verify the install { display-mode: "form" }
#@markdown Runs inside the Helixer environment: imports, GPU visibility, and a Conv1D + cuDNN-LSTM smoke test
#@markdown (the layer types Helixer's models use), plus HelixerPost and gffread.
import json, subprocess

PROBE = r"""
import json, importlib.metadata as m, tensorflow as tf
def ver(p):
    try:
        return m.version(p)
    except Exception:
        return "not installed"
gpus = tf.config.list_physical_devices("GPU")
smoke = "skipped (no GPU)"
if gpus:
    with tf.device("/GPU:0"):
        x = tf.random.normal((2, 999, 4))
        y = tf.keras.layers.Conv1D(16, 9, padding="same")(x)
        y = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True))(y)
        smoke = "ok" if bool(tf.reduce_all(tf.math.is_finite(y))) else "NaN output"
bi = tf.sysconfig.get_build_info()
print(json.dumps({"helixer": ver("helixer"), "geenuff": ver("geenuff"), "tensorflow": tf.__version__,
                  "keras": ver("keras"), "tf-addons": ver("tensorflow-addons"),
                  "cuda (built)": bi.get("cuda_version"), "cudnn (built)": bi.get("cudnn_version"),
                  "GPUs": len(gpus), "GPU smoke test": smoke}))
"""
res = subprocess.run([HELIXER_PY, "-c", PROBE], capture_output=True, text=True, env=HELIXER_ENV)
if res.returncode != 0:
    print(res.stdout); print(res.stderr[-4000:])
    raise RuntimeError("The Helixer environment failed the probe. Delete /content/helixer-env and re-run cell 2.")
info = json.loads(res.stdout.strip().splitlines()[-1])
for k, v in info.items():
    print(f"{k:<16}: {v}")

post = subprocess.run([POST_BIN], capture_output=True, text=True)
assert post.returncode == 1 and "HelixerPost" in (post.stdout + post.stderr), "helixer_post_bin is not working"
print(f"{'helixer_post_bin':<16}: ok ({POST_BIN})")
print(f"{'gffread':<16}:", subprocess.run(["gffread", "--version"], capture_output=True, text=True).stdout.strip() or "MISSING")

if GPU_CC and not info["GPUs"]:
    raise RuntimeError("A GPU is attached but TensorFlow 2.15 cannot see it - check the stderr above "
                       "(delete /content/helixer-env and re-run cell 2).")
if info["GPUs"] and info["GPU smoke test"] != "ok":
    raise RuntimeError(f"GPU smoke test: {info['GPU smoke test']}")
print("\nHelixer environment OK" + ("" if info["GPUs"] else " (CPU only)"))

In [ ]:
#@title 4 · Storage — mount Drive & start a run folder { display-mode: "form" }
#@markdown Mount Google Drive so inputs, the models and outputs persist across sessions.
#@markdown Every run gets its own timestamped folder under `WORKDIR/runs/`, so nothing is overwritten.
MOUNT_DRIVE = True  #@param {type:"boolean"}
WORKDIR = "/content/drive/MyDrive/helixer_runs"  #@param {type:"string"}
RUN_TAG = ""  #@param {type:"string"}
TIMEZONE = "Europe/London"  #@param {type:"string"}
#@markdown `RUN_TAG` is an optional suffix for the folder name (e.g. `oomycetes`). `TIMEZONE` only
#@markdown affects the timestamp; the Colab clock itself is UTC.
RESUME_RUN = ""  #@param {type:"string"}
#@markdown To **continue an interrupted batch** in a new session, paste the existing run folder name here
#@markdown (e.g. `run_20261001-142530_oomycetes`); with `SKIP_DONE` (cell 5) finished genomes are skipped.

import os, re
from datetime import datetime, timezone as _tz

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif WORKDIR.startswith("/content/drive"):
    WORKDIR = "/content/work"

try:
    from zoneinfo import ZoneInfo
    NOW = datetime.now(ZoneInfo(TIMEZONE))
except Exception:
    NOW, TIMEZONE = datetime.now(_tz.utc), "UTC"

tag = re.sub(r"[^A-Za-z0-9._-]+", "_", RUN_TAG.strip()).strip("_")
if RESUME_RUN.strip():
    RUN_ID = os.path.basename(RESUME_RUN.strip().rstrip("/"))
    assert os.path.isdir(os.path.join(WORKDIR, "runs", RUN_ID)), \
        f"No run folder {RUN_ID} under {os.path.join(WORKDIR, 'runs')}"
    print("Resuming existing run folder.")
else:
    RUN_ID = NOW.strftime("run_%Y%m%d-%H%M%S") + (f"_{tag}" if tag else "")
RUNDIR = os.path.join(WORKDIR, "runs", RUN_ID)
CKPT_ROOT = os.path.join(WORKDIR, "helixer_models")     # shared across runs, downloaded once
os.makedirs(RUNDIR, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

print("Working directory:", WORKDIR)
print("Run folder       :", RUNDIR, f"({TIMEZONE})")
print("Model cache      :", CKPT_ROOT)
print("\nRe-run this cell to start a new run folder; re-running cells 8-13 alone keeps writing to this one.")

In [ ]:
#@title 5 · Prediction parameters & genome input { display-mode: "form" }
#@markdown **Model** — pick the lineage your species belongs to.
LINEAGE = "land_plant"  #@param ["land_plant", "vertebrate", "invertebrate", "fungi"]
MODEL_FILE = "best"  #@param ["best"] {allow-input: true}
#@markdown `best` = the top-priority model for the lineage in Helixer's model list (e.g. `land_plant_v0.3_a_0080.h5`). Type another file name from that list (printed by cell 6) to use it instead.
SPECIES = "Helixer"  #@param {type:"string"}
#@markdown Written to the GFF3 header **and used as the gene-ID prefix** (`Helixer_<seqid>_000001`). Leave empty to use each genome's file name.
SUBSEQUENCE_LENGTH = "auto"  #@param ["auto", "21384", "64152", "106920", "213840"] {allow-input: true}
#@markdown `auto` = lineage default (fungi 21384, land_plant 64152, vertebrate / invertebrate 213840). Raise it for genomes with long genic loci (plants: try 106920); keep it below the assembly's N50. Must be divisible by 9.
OVERLAP = True  #@param {type:"boolean"}
#@markdown Sliding-window overlap (offset = length/2, core = 3/4 length). Better gene models at subsequence ends; ~2x the prediction time.
PEAK_THRESHOLD = 0.8  #@param {type:"number"}
#@markdown Precision / recall knob of HelixerPost: 0.9 - 0.975 raises precision with very little BUSCO loss.
EDGE_THRESHOLD = 0.1  #@param {type:"number"}
WINDOW_SIZE = 100  #@param {type:"integer"}
MIN_CODING_LENGTH = 60  #@param {type:"integer"}
BATCH_SIZE = "32"  #@param ["8", "16", "32", "64", "128"] {allow-input: true}
#@markdown 32 is Helixer's default and fits a T4 for every lineage. Lower it on OOM; raise it on an A100/H100.
DETERMINISTIC = False  #@param {type:"boolean"}
#@markdown Forces deterministic cuDNN/cuBLAS kernels — bit-identical reruns on the same GPU type, possibly slower.
WANT_CODING = True  #@param {type:"boolean"}
WANT_PROTEIN = True  #@param {type:"boolean"}
SKIP_DONE = True  #@param {type:"boolean"}
#@markdown `SKIP_DONE`: genomes that already have a GFF3 in **this run folder** are not re-predicted — re-run cell 8 after a disconnect to resume a batch.
STOP_ON_ERROR = False  #@param {type:"boolean"}
#@markdown Off: a failed genome is logged and the batch moves on (failures are listed at the end of cell 8).
#@markdown ---
#@markdown **Genome input** — `drive_dir` runs on **every** FASTA found recursively under the folder; the others take a single genome.
GENOME_SOURCE = "drive_dir"  #@param ["drive_dir", "upload", "url", "drive_file"]
GENOME_DIR = "/content/drive/MyDrive/genomes"  #@param {type:"string"}
GENOME_PATH = "/content/drive/MyDrive/genome.fasta"  #@param {type:"string"}
GENOME_URL = ""  #@param {type:"string"}

LINEAGE_SUBSEQ = {"fungi": 21384, "land_plant": 64152, "vertebrate": 213840, "invertebrate": 213840}
SUBSEQ = None if str(SUBSEQUENCE_LENGTH).strip() == "auto" else int(SUBSEQUENCE_LENGTH)   # resolved in cell 6
if SUBSEQ is not None:
    assert SUBSEQ > 0 and SUBSEQ % 9 == 0, "SUBSEQUENCE_LENGTH must be a positive multiple of 9 (the model's timestep width)."
assert 0 < EDGE_THRESHOLD < PEAK_THRESHOLD <= 1, "Need 0 < EDGE_THRESHOLD < PEAK_THRESHOLD <= 1."
assert WINDOW_SIZE > 0 and MIN_CODING_LENGTH >= 0
BATCH = int(BATCH_SIZE)
SPECIES = SPECIES.strip().replace(" ", "_")

print(f"lineage {LINEAGE} | model {MODEL_FILE} | species {SPECIES or '<genome name>'} | "
      f"subseq {SUBSEQ or 'auto'} | overlap {OVERLAP} | batch {BATCH} | source {GENOME_SOURCE}")
print(f"post-processing: window {WINDOW_SIZE} | edge {EDGE_THRESHOLD} | peak {PEAK_THRESHOLD} | min CDS {MIN_CODING_LENGTH} bp")

In [ ]:
#@title 6 · Download the lineage model { display-mode: "form" }
#@markdown Reads Helixer's live model list and downloads only the selected model (~25 MB, from Zenodo) into `WORKDIR/helixer_models/<lineage>/`. It sits outside the run folders, so it is downloaded once and reused by every later run.
import os, csv, hashlib, requests

LIST_URL = "https://raw.githubusercontent.com/usadellab/Helixer/main/resources/model_list.csv"
try:
    r = requests.get(LIST_URL, timeout=60); r.raise_for_status()
    MODEL_LIST_TXT, src = r.text, LIST_URL
except Exception as e:
    MODEL_LIST_TXT, src = open(os.path.join(REPO, "resources", "model_list.csv")).read(), "bundled copy (offline)"
models = list(csv.DictReader(l for l in MODEL_LIST_TXT.splitlines() if l.strip() and not l.startswith("#")))
for m_ in models:
    m_["priority"] = int(m_["priority"])
lineage_models = sorted((m_ for m_ in models if m_["lineage"] == LINEAGE), key=lambda m_: m_["priority"])
assert lineage_models, f"No models listed for lineage {LINEAGE}"

want = MODEL_FILE.strip()
if want in ("", "best"):
    MODEL = lineage_models[0]
else:
    want = want if want.endswith(".h5") else want + ".h5"
    MODEL = next((m_ for m_ in models if m_["model_file_name"] == want), None)
    assert MODEL, f"'{want}' is not in the model list. Options for {LINEAGE}: " + \
                  ", ".join(m_["model_file_name"] for m_ in lineage_models)
    if MODEL["lineage"] != LINEAGE:
        print(f"NOTE: {want} is a {MODEL['lineage']} model (LINEAGE is {LINEAGE}); using its lineage defaults.")

MODEL_PATH = os.path.join(CKPT_ROOT, MODEL["lineage"], MODEL["model_file_name"])
if os.path.exists(MODEL_PATH):
    print("already present:", MODEL_PATH)
else:
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    tmp = MODEL_PATH + ".part"
    with requests.get(MODEL["download_link"], stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
    with open(tmp, "rb") as fh:
        assert fh.read(8) == b"\x89HDF\r\n\x1a\n", "Download is not an HDF5 model (Zenodo error page?) - re-run this cell."
    os.replace(tmp, MODEL_PATH)
with open(os.path.join(CKPT_ROOT, "model_list.csv"), "w") as fh:
    fh.write(MODEL_LIST_TXT)

MODEL_MD5 = hashlib.md5(open(MODEL_PATH, "rb").read()).hexdigest()
if SUBSEQ is None:
    SUBSEQ = LINEAGE_SUBSEQ[MODEL["lineage"]]

lineage_models = sorted((m_ for m_ in models if m_["lineage"] == MODEL["lineage"]), key=lambda m_: m_["priority"])
print(f"\nmodel list: {src}")
print(f"{'':2}{'model':<32}{'released':<12}priority")
for m_ in lineage_models:
    mark = "->" if m_["model_file_name"] == MODEL["model_file_name"] else ""
    print(f"{mark:<2}{m_['model_file_name']:<32}{m_['available_since']:<12}{m_['priority']}")
print(f"\nusing {MODEL['model_file_name']}  ({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB, md5 {MODEL_MD5})")
print(f"subsequence length {SUBSEQ}" + (f" | overlap offset {SUBSEQ // 2}, core {SUBSEQ * 3 // 4}" if OVERLAP else ""))

In [ ]:
#@title 7 · Collect genome(s) { display-mode: "form" }
import os, shutil, gzip, hashlib, subprocess

STAGE = "/content/run"; GEN_DIR = os.path.join(STAGE, "genomes"); os.makedirs(GEN_DIR, exist_ok=True)
FASTA_EXT = (".fa", ".fasta", ".fna", ".fa.gz", ".fasta.gz", ".fna.gz")

def base_of(p):
    b = os.path.basename(p)
    for e in (".fa.gz", ".fasta.gz", ".fna.gz", ".fa", ".fasta", ".fna"):
        if b.lower().endswith(e):
            return b[:-len(e)]
    return os.path.splitext(b)[0]

def stage_local(src, base):
    # Helixer reads .gz directly, but gffread needs a plain FASTA - keep one uncompressed local copy.
    # The staged name is salted with the source path so same-named genomes in different
    # subfolders cannot overwrite each other.
    salt = hashlib.md5(os.path.abspath(src).encode()).hexdigest()[:6]
    out = os.path.join(GEN_DIR, f"{base}__{salt}.fasta")
    if os.path.exists(out):
        return out
    if src.endswith(".gz"):
        with gzip.open(src, "rb") as fi, open(out, "wb") as fo:
            shutil.copyfileobj(fi, fo)
    else:
        shutil.copy(src, out)
    return out

raw = []
if GENOME_SOURCE == "drive_dir":
    assert MOUNT_DRIVE, "Enable MOUNT_DRIVE in cell 4 to read from Drive."
    assert os.path.isdir(GENOME_DIR), f"Not a directory: {GENOME_DIR}"
    for root, _, fnames in os.walk(GENOME_DIR):
        for fn in fnames:
            if fn.lower().endswith(FASTA_EXT):
                raw.append(os.path.join(root, fn))
    assert raw, f"No FASTA (.fa/.fasta/.fna[.gz]) found under {GENOME_DIR}"
elif GENOME_SOURCE == "drive_file":
    assert os.path.exists(GENOME_PATH), f"Not found: {GENOME_PATH}"
    raw = [GENOME_PATH]
elif GENOME_SOURCE == "url":
    assert GENOME_URL, "Set GENOME_URL in cell 5."
    dst = os.path.join(GEN_DIR, GENOME_URL.split("/")[-1].split("?")[0])
    subprocess.run(["wget", "-q", "-O", dst, GENOME_URL], check=True)
    raw = [dst]
elif GENOME_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    for name in up:
        d = os.path.join(GEN_DIR, name); os.replace(name, d); raw.append(d)

GENOMES = [(base_of(p), stage_local(p, base_of(p))) for p in sorted(raw)]
dupes = {b for b, _ in GENOMES if [x for x, _ in GENOMES].count(b) > 1}
assert not dupes, f"Genome names must be unique - these appear more than once: {sorted(dupes)}"

print(f"{len(GENOMES)} genome(s) staged:")
for b, p in GENOMES:
    print(f"   {b:<40}{os.path.getsize(p) / 1e6:8.1f} MB   {p}")

In [ ]:
#@title 8 · Run Helixer + gffread on all genome(s) { display-mode: "form" }
#@markdown Per genome: `Helixer.py` (FASTA -> H5 -> neural network -> HelixerPost, run with the Helixer environment) -> GFF3, then `gffread` -> CDS / protein. Everything is gzipped into `helixer_predictions/`.
import os, re, gzip, json, shutil, subprocess, time
from datetime import datetime

PRED_LOCAL = "/content/run/pred"; LOG_LOCAL = "/content/run/log"; TMP_LOCAL = "/content/run/tmp"
for d in (PRED_LOCAL, LOG_LOCAL, TMP_LOCAL):
    os.makedirs(d, exist_ok=True)

OUTBASE = os.path.join(RUNDIR, "helixer_predictions")
DIRS = {k: os.path.join(OUTBASE, k) for k in ("gff", "cds", "protein", "log")}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# a record of exactly what produced this run folder (a resumed run adds a second, timestamped record)
cfg_path = os.path.join(RUNDIR, "run_config.json")
if os.path.exists(cfg_path):
    cfg_path = os.path.join(RUNDIR, f"run_config_resumed_{datetime.now(NOW.tzinfo):%Y%m%d-%H%M%S}.json")
with open(cfg_path, "w") as fh:
    json.dump({"run_id": RUN_ID, "started": NOW.isoformat(), "timezone": TIMEZONE,
               "helixer_version": HELIXER_VERSION, "lineage": MODEL["lineage"],
               "model": MODEL["model_file_name"], "model_md5": MODEL_MD5, "model_path": MODEL_PATH,
               "species": SPECIES or "<genome name>", "subsequence_length": SUBSEQ, "overlap": OVERLAP,
               "overlap_offset": SUBSEQ // 2, "overlap_core_length": SUBSEQ * 3 // 4,
               "batch_size": BATCH, "deterministic": DETERMINISTIC, "window_size": WINDOW_SIZE,
               "edge_threshold": EDGE_THRESHOLD, "peak_threshold": PEAK_THRESHOLD,
               "min_coding_length": MIN_CODING_LENGTH, "genomes": [b for b, _ in GENOMES],
               "environment": info}, fh, indent=2)

def gzip_to(src, dst_dir, level=6):
    dst = os.path.join(dst_dir, os.path.basename(src) + ".gz")
    with open(src, "rb") as fi, gzip.open(dst, "wb", compresslevel=level) as fo:
        shutil.copyfileobj(fi, fo)
    return dst

def gunzip_to(src_gz, dst):
    with gzip.open(src_gz, "rb") as fi, open(dst, "wb") as fo:
        shutil.copyfileobj(fi, fo)
    return dst

def run_logged(cmd, logpath, env=None, cwd=None):
    # stream stdout+stderr to the console and a log file simultaneously
    with open(logpath, "w") as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env, cwd=cwd)
        for line in p.stdout:
            print(line, end=""); log.write(line)
        p.wait()
    return p.returncode

def has_genes(gff):
    if not os.path.exists(gff):
        return False
    with open(gff) as fh:
        return any("\tgene\t" in ln for ln in fh)

JOBS, FAILED = [], []
for i, (base, gpath) in enumerate(GENOMES, 1):
    print(f"\n===== [{i}/{len(GENOMES)}] {base} =====")
    gff  = os.path.join(PRED_LOCAL, f"{base}_gff.gff3")
    cds  = os.path.join(PRED_LOCAL, f"{base}_cds.fasta")
    prot = os.path.join(PRED_LOCAL, f"{base}_protein.fasta")
    job = {"base": base, "genome": gpath, "gff": gff, "cds": cds, "protein": prot}
    gff_gz = os.path.join(DIRS["gff"], os.path.basename(gff) + ".gz")
    prot_gz = os.path.join(DIRS["protein"], os.path.basename(prot) + ".gz")

    if SKIP_DONE and os.path.exists(gff_gz) and (not WANT_PROTEIN or os.path.exists(prot_gz)):
        if WANT_PROTEIN and not os.path.exists(prot):
            gunzip_to(prot_gz, prot)                 # BUSCO (cell 12) reads the plain protein FASTA
        print("already done in this run folder - skipped (SKIP_DONE)")
        JOBS.append(job); continue

    log_local = os.path.join(LOG_LOCAL, f"{base}.log")
    cmd = [HELIXER_PY, os.path.join(ENV_BIN, "Helixer.py"),
           "--fasta-path", gpath, "--gff-output-path", gff,
           "--species", SPECIES or re.sub(r"[^A-Za-z0-9._-]+", "_", base),
           "--model-filepath", MODEL_PATH, "--subsequence-length", str(SUBSEQ),
           "--batch-size", str(BATCH), "--temporary-dir", TMP_LOCAL,
           "--window-size", str(WINDOW_SIZE), "--edge-threshold", str(EDGE_THRESHOLD),
           "--peak-threshold", str(PEAK_THRESHOLD), "--min-coding-length", str(MIN_CODING_LENGTH)]
    if not OVERLAP:    cmd.append("--no-overlap")
    if DETERMINISTIC:  cmd.append("--deterministic")
    print(" ".join(cmd))
    t0 = time.time()
    rc = run_logged(cmd, log_local, env=HELIXER_ENV, cwd=TMP_LOCAL)
    shutil.copy(log_local, DIRS["log"])                      # keep the log even on failure
    # Helixer.py exits 0 even when HelixerPost fails, so success = a GFF3 with genes in it
    if rc != 0 or not has_genes(gff):
        msg = f"Helixer failed on {base} (exit {rc}{', no genes in GFF3' if rc == 0 else ''}); see {os.path.join(DIRS['log'], base + '.log')}"
        if STOP_ON_ERROR:
            raise RuntimeError(msg)
        print("ERROR:", msg); FAILED.append(base); continue

    saved = [gzip_to(gff, DIRS["gff"])]
    if WANT_CODING or WANT_PROTEIN:
        gcmd = ["gffread", gff, "-g", gpath]
        if WANT_CODING:  gcmd += ["-x", cds]
        if WANT_PROTEIN: gcmd += ["-y", prot, "-S"]
        gr = subprocess.run(gcmd, capture_output=True, text=True)
        if gr.returncode != 0:
            print("gffread WARNING:", gr.stderr.strip()[:2000])
    if WANT_CODING and os.path.exists(cds):    saved.append(gzip_to(cds, DIRS["cds"]))
    if WANT_PROTEIN and os.path.exists(prot):  saved.append(gzip_to(prot, DIRS["protein"]))
    print(f"saved ({(time.time() - t0) / 60:.1f} min):", ", ".join(os.path.relpath(s, OUTBASE) for s in saved))
    JOBS.append(job)

shutil.rmtree(TMP_LOCAL, ignore_errors=True)
print(f"\nDone. {len(JOBS)} prediction(s) under {OUTBASE}/  (gff/ cds/ protein/ log/)")
if FAILED:
    print(f"FAILED ({len(FAILED)}):", ", ".join(FAILED), "- see their logs in", DIRS["log"])
print("Run folder:", RUNDIR)

In [ ]:
#@title 9 · Summary of predictions { display-mode: "form" }
import os, gzip, statistics as st
from collections import Counter
import pandas as pd
from IPython.display import display

PRED_OUT = os.path.join(RUNDIR, "helixer_predictions")

def _resolve(local, subdir):
    # prefer the local uncompressed file; fall back to the gzipped copy on Drive
    if os.path.exists(local):
        return local
    gz = os.path.join(PRED_OUT, subdir, os.path.basename(local) + ".gz")
    return gz if os.path.exists(gz) else local

def _open(p):
    return gzip.open(p, "rt") if p.endswith(".gz") else open(p)

def gff_stats(path):
    feat = Counter(); cds_bp = 0
    if not os.path.exists(path):
        return feat, cds_bp
    with _open(path) as fh:
        for ln in fh:
            if ln.startswith("#"):
                continue
            f = ln.rstrip("\n").split("\t")
            if len(f) < 8:
                continue
            feat[f[2].lower()] += 1
            if f[2].lower() == "cds":
                cds_bp += int(f[4]) - int(f[3]) + 1
    return feat, cds_bp

def prot_lengths(fa):
    L = []; cur = 0
    if not os.path.exists(fa):
        return L
    with _open(fa) as fh:
        for ln in fh:
            if ln.startswith(">"):
                if cur:
                    L.append(cur)
                cur = 0
            else:
                cur += sum(1 for ch in ln.strip() if ch not in "*.")
    if cur:
        L.append(cur)
    return L

def genome_bp(fa):
    fai = fa + ".fai"                          # written by gffread in cell 8
    if os.path.exists(fai):
        return sum(int(l.split("\t")[1]) for l in open(fai))
    n = 0
    with open(fa) as fh:
        for ln in fh:
            if not ln.startswith(">"):
                n += len(ln.strip())
    return n

rows = []
for j in JOBS:
    feat, cds_bp = gff_stats(_resolve(j["gff"], "gff"))
    L = prot_lengths(_resolve(j["protein"], "protein"))
    mrna = feat.get("mrna", 0)
    gbp = genome_bp(j["genome"])
    rows.append({
        "genome": j["base"],
        "genome_Mb": round(gbp / 1e6, 1),
        "genes": feat.get("gene", 0),
        "genes/Mb": round(feat.get("gene", 0) / (gbp / 1e6), 1) if gbp else 0,
        "mRNA": mrna,
        "proteins": len(L),
        "CDS": feat.get("cds", 0),
        "exons": feat.get("exon", 0),
        "exons/mRNA": round(feat.get("exon", 0) / mrna, 2) if mrna else 0,
        "5UTR": feat.get("five_prime_utr", 0),
        "3UTR": feat.get("three_prime_utr", 0),
        "mean_aa": round(st.mean(L)) if L else 0,
        "median_aa": round(st.median(L)) if L else 0,
        "max_aa": max(L) if L else 0,
        "total_CDS_Mb": round(cds_bp / 1e6, 2),
    })

df = pd.DataFrame(rows)
out_tsv = os.path.join(RUNDIR, "prediction_summary.tsv")
df.to_csv(out_tsv, sep="\t", index=False)
display(df)
print("Saved:", out_tsv)
print("Outputs under:", PRED_OUT, " (gff/ cds/ protein/ log/)")

## BUSCO completeness

Two independent checks, each toggled in the next cell:
- **proteins mode** — runs on Helixer's predicted proteins; how complete the **annotation** is. HMMER only -> fast (needs `WANT_PROTEIN = True` in cell 5).
- **genome mode** — runs on each assembly via miniprot; the assembly's completeness **ceiling**, independent of annotation. Needs bbtools + miniprot and is **much slower** (CPU-bound — for whole genomes prefer HPC).

Pick a lineage matching the clade of your genomes, or choose **other** and type any dataset name. List all with `busco --list-datasets`. Every genome in the batch is scored against the same lineage.

In [ ]:
#@title 10 · BUSCO options { display-mode: "form" }
RUN_PROTEIN_BUSCO = True   #@param {type:"boolean"}
RUN_GENOME_BUSCO = False   #@param {type:"boolean"}
BUSCO_LINEAGE = "eudicotyledons_odb12.2"  #@param ["eukaryota_odb12.2", "viridiplantae_odb12.2", "embryophyta_odb12.2", "eudicotyledons_odb12.2", "liliopsida_odb12.2", "chlorophyta_odb12.2", "stramenopiles_odb12.2", "oomycota_odb12.2", "fungi_odb12.2", "ascomycota_odb12.2", "basidiomycota_odb12.2", "sordariomycetes_odb12.2", "saccharomycetes_odb12.2", "metazoa_odb12.2", "arthropoda_odb12.2", "insecta_odb12.2", "lepidoptera_odb12.2", "nematoda_odb12.2", "mollusca_odb12.2", "vertebrata_odb12.2", "mammalia_odb12.2", "aves_odb12.2", "actinopterygii_odb12.2", "other"] {allow-input: true}
#@markdown If `BUSCO_LINEAGE` is **other**, type the dataset name here:
BUSCO_LINEAGE_CUSTOM = ""  #@param {type:"string"}

LINEAGE_BUSCO = BUSCO_LINEAGE_CUSTOM.strip() if BUSCO_LINEAGE == "other" else BUSCO_LINEAGE
assert LINEAGE_BUSCO and LINEAGE_BUSCO != "other", "Choose a lineage, or set BUSCO_LINEAGE_CUSTOM when using 'other'."
print("lineage:", LINEAGE_BUSCO)

In [ ]:
#@title 11 · Install BUSCO (+ genome-mode tools if enabled) { display-mode: "form" }
#@markdown HMMER + BUSCO for both modes. Genome mode also builds miniprot and installs bbtools. ~1-3 min.
import os, subprocess
subprocess.call("apt-get -qq install -y hmmer > /dev/null", shell=True)
if not os.path.isdir("/content/busco"):
    subprocess.call("git clone -q --depth 1 https://gitlab.com/ezlab/busco.git /content/busco", shell=True)
subprocess.call("pip install -q /content/busco biopython pandas requests scipy", shell=True)

if RUN_GENOME_BUSCO:
    subprocess.call("apt-get -qq install -y bbmap > /dev/null", shell=True)
    subprocess.call("ln -sf /usr/share/bbmap/stats.sh /usr/bin/stats.sh", shell=True)
    subprocess.call("ln -sf /usr/share/bbmap/statswrapper.sh /usr/bin/statswrapper.sh", shell=True)
    if not os.path.exists("/usr/local/bin/miniprot"):
        subprocess.call("git clone -q --depth 1 https://github.com/lh3/miniprot /content/miniprot", shell=True)
        subprocess.call("make -C /content/miniprot -j2 && cp /content/miniprot/miniprot /usr/local/bin/", shell=True)
    print("miniprot:", subprocess.run(["miniprot", "--version"], capture_output=True, text=True).stdout.strip())
print("busco   :", subprocess.run(["busco", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 12 · Run BUSCO + compile results table { display-mode: "form" }
import os, glob, re, csv, gzip, shutil, subprocess
BUSCO_LOCAL = "/content/busco_runs"; os.makedirs(BUSCO_LOCAL, exist_ok=True)
assert RUN_PROTEIN_BUSCO or RUN_GENOME_BUSCO, "Both BUSCO toggles are off (cell 10)."

LIN_TAG = re.sub(r"[^A-Za-z0-9._-]+", "_", LINEAGE_BUSCO)      # goes into every output name

def parse_summary(path):
    m = re.search(r"C:([\d.]+)%\[S:([\d.]+)%,D:([\d.]+)%\],F:([\d.]+)%,M:([\d.]+)%,n:(\d+)", open(path).read())
    keys = ["C%", "S%", "D%", "F%", "M%", "n"]
    return dict(zip(keys, m.groups())) if m else {k: "" for k in keys}

def local_protein(j):
    # the plain protein FASTA, restored from the gzipped Drive copy after a runtime reset
    if not os.path.exists(j["protein"]):
        gz = os.path.join(RUNDIR, "helixer_predictions", "protein", os.path.basename(j["protein"]) + ".gz")
        if os.path.exists(gz):
            os.makedirs(os.path.dirname(j["protein"]), exist_ok=True)
            with gzip.open(gz, "rb") as fi, open(j["protein"], "wb") as fo:
                shutil.copyfileobj(fi, fo)
    return j["protein"]

rows = []
BUSCO_RUNS = []          # directory names produced by this cell, zipped by cell 13
for j in JOBS:
    targets = []
    if RUN_PROTEIN_BUSCO:
        prot = local_protein(j)
        assert os.path.exists(prot), f"No protein FASTA for {j['base']} - enable WANT_PROTEIN and re-run cell 8."
        targets.append(("proteins", prot, ["-m", "proteins"], f"{j['base']}_busco_prot_{LIN_TAG}"))
    if RUN_GENOME_BUSCO:
        targets.append(("genome", j["genome"], ["-m", "genome", "--miniprot"], f"{j['base']}_busco_genome_{LIN_TAG}"))
    for mode, inp, modeargs, name in targets:
        cmd = ["busco", "-i", inp, *modeargs, "-l", LINEAGE_BUSCO, "-o", name,
               "--out_path", BUSCO_LOCAL, "-c", str(os.cpu_count() or 2), "-f"]
        print(f"\n##### BUSCO {mode}: {j['base']} #####")
        subprocess.run(cmd, check=True)
        summ = glob.glob(os.path.join(BUSCO_LOCAL, name, "short_summary*.txt"))
        rows.append({"genome": j["base"], "mode": mode, "lineage": LINEAGE_BUSCO,
                     **(parse_summary(summ[0]) if summ else {})})
        BUSCO_RUNS.append(name)

cols = ["genome", "mode", "lineage", "C%", "S%", "D%", "F%", "M%", "n"]
BUSCO_OUT = os.path.join(RUNDIR, "helixer_predictions", "busco")
os.makedirs(BUSCO_OUT, exist_ok=True)
table = os.path.join(BUSCO_OUT, "busco_summary.tsv")

# one table per run folder, accumulating across lineages: rows are keyed by
# (genome, mode, lineage), so re-running a lineage replaces its rows instead of duplicating them
key = lambda r: (r["genome"], r["mode"], r["lineage"])
merged = {}
if os.path.exists(table):
    with open(table) as f:
        for r in csv.DictReader(f, delimiter="\t"):
            merged[key(r)] = r
for r in rows:
    merged[key(r)] = r
with open(table, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t"); w.writeheader()
    w.writerows([merged[k] for k in sorted(merged)])

print("\n=== BUSCO summary table ===")
print("\t".join(cols))
for k in sorted(merged):
    print("\t".join(str(merged[k].get(c, "")) for c in cols))
print(f"\nTable saved: {table}  ({len(merged)} row(s), {len({k[2] for k in merged})} lineage(s))")

In [ ]:
#@title 13 · Compress & save BUSCO run directories { display-mode: "form" }
import os, glob, shutil
BUSCO_LOCAL = "/content/busco_runs"
BUSCO_OUT = os.path.join(RUNDIR, "helixer_predictions", "busco"); os.makedirs(BUSCO_OUT, exist_ok=True)

# only the runs cell 12 just produced, so earlier lineages in this session are not re-zipped
todo = [os.path.join(BUSCO_LOCAL, n) for n in BUSCO_RUNS] if "BUSCO_RUNS" in dir() \
       else sorted(glob.glob(os.path.join(BUSCO_LOCAL, "*")))
made = []
for d in todo:
    if os.path.isdir(d):
        arc = shutil.make_archive(os.path.join(BUSCO_OUT, os.path.basename(d)), "zip", d)
        made.append(arc); print("zipped:", os.path.basename(arc))
if not MOUNT_DRIVE:
    from google.colab import files
    for z in made:
        files.download(z)
print("\nBUSCO zips and busco_summary.tsv in:", BUSCO_OUT)
print("Zips present:", len(glob.glob(os.path.join(BUSCO_OUT, "*.zip"))))

---
### Notes & troubleshooting
- **Outputs** land in a timestamped run folder, `WORKDIR/runs/run_<date>-<time>[_tag]/`: predictions in `helixer_predictions/` split into `gff/`, `cds/`, `protein/` (all gzipped), `log/` (per-genome Helixer log) and `busco/` (one zip per BUSCO run plus `busco_summary.tsv`); alongside them `prediction_summary.tsv` and `run_config.json` (model + md5, every parameter, and package versions). Models live outside the run folders in `WORKDIR/helixer_models/` and are downloaded once.
- **Starting a new run:** re-run cell 4 — it stamps a fresh folder. Re-running cells 8-13 on their own keeps writing into the current one.
- **Resuming a batch after a disconnect:** in the new session set `RESUME_RUN` in cell 4 to the interrupted run folder's name and run cells 1-8 again. With `SKIP_DONE` on, genomes that already have a GFF3 there are skipped (if the session survived, just re-run cell 8).
- **Gene IDs:** Helixer builds them from `--species`: `<SPECIES>_<seqid>_<6-digit index>` with transcripts `.1` and features `.exon.N` / `.CDS.N`. Proteins and CDS carry the transcript ID.
- **Softmasking is ignored:** Helixer encodes lowercase and uppercase bases identically, so RefSeq/Ensembl softmasked genomes can go in as-is. Do not hardmask — the models were trained on unmasked sequence.
- **Model choice matters most.** `--subsequence-length` is the second knob: it should comfortably exceed typical gene-locus length but stay below the N50 (ideally N90). Very fragmented assemblies with a long subsequence length waste compute on padding.
- **`PEAK_THRESHOLD`:** 0.8 is the default; 0.9 / 0.95 / 0.975 trade a little recall for noticeably fewer spurious genes.
- **GPU OOM:** lower `BATCH_SIZE` in cell 5 (and/or `SUBSEQUENCE_LENGTH`).
- **The Helixer environment:** cell 2 builds `/content/helixer-env` (Python 3.10, TensorFlow 2.15.1, Keras 2.15, tensorflow-addons, CUDA 12.2 / cuDNN 8.9 pip wheels) and compiles `helixer_post_bin` into it. Colab's own Python is never touched and no restart is needed. It is rebuilt each session; re-running cell 2 is a no-op once it imports cleanly. If cell 3 fails, delete `/content/helixer-env` and re-run cell 2.
- **Blackwell GPUs** (compute capability 10.x/12.x) postdate the pinned TensorFlow/cuDNN; if cell 3's smoke test fails there, use a T4 / L4 / A100 / H100 runtime.
- **CPU-only runtimes work** but are only practical for a few Mb of sequence.
- **Reproducibility:** results are deterministic on most GPUs; tick `DETERMINISTIC` on GPUs whose default cuDNN kernels are not. Predictions can still differ slightly between GPU architectures.
- **Helixer exits 0 even when post-processing fails**, so cell 8 treats a GFF3 without genes as a failure and points to the log.
- **BUSCO + Drive:** BUSCO uses symlinks (unsupported on the Drive mount), so it always runs locally (`/content/busco_runs`); cell 13 zips each run into `helixer_predictions/busco/`, next to `busco_summary.tsv`. Change `BUSCO_LINEAGE` and re-run cells 10-13 to add another lineage to the same table.
- **Genome BUSCO is slow** on Colab cores — fine for scaffolds, better on HPC for whole genomes.
- **Edit a hidden cell:** menu (vertical dots) > *Show code*.
- Cite: Holst *et al.* (2025) *Nat Methods* doi:10.1038/s41592-025-02939-1; Stiehler *et al.* (2020) *Bioinformatics* doi:10.1093/bioinformatics/btaa1044; Manni *et al.* (2021) for BUSCO.